# Step 14: Round 1j Patchback Summary

将 `Round 1g` 与 `Round 1i` 的 repaired relation-chain evidence 补回 `Round 1c` 的 role-aware interpretation，重点只更新 `wiki_dev_2639` 的诊断含义。


In [ ]:
import csv
from pathlib import Path

PROJECT_ROOT_OVERRIDE = ''

def candidate_roots():
    candidates = []
    if PROJECT_ROOT_OVERRIDE.strip():
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / '2026_SelectTransfer',
        Path('/content/2026_SelectTransfer'),
        Path('/workspace/2026_SelectTransfer'),
        Path('/root/2026_SelectTransfer'),
        Path('/Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer'),
    ])
    out = []
    seen = set()
    for c in candidates:
        if c in seen:
            continue
        seen.add(c)
        out.append(c)
    return out


def detect_project_root():
    checked = []
    for candidate in candidate_roots():
        checked.append(str(candidate))
        if (candidate / 'results' / '05_round1b_prep' / 'round1c_role_aware_smoke_table.csv').exists() and (candidate / 'results' / '12_round1i_run' / 'round1i_operator_results.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root. Checked: ' + ' | '.join(checked))

PROJECT_ROOT = detect_project_root()
RESULTS_DIR = PROJECT_ROOT / 'results' / '13_round1j_summary'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ROUND1C_TABLE = PROJECT_ROOT / 'results' / '05_round1b_prep' / 'round1c_role_aware_smoke_table.csv'
ROUND1C_SUMMARY_MD = PROJECT_ROOT / 'results' / '06_round1c_summary' / 'round1c_allowed_aggregate_summary.md'
ROUND1G_RESULTS = PROJECT_ROOT / 'results' / '10_round1g_run' / 'round1g_relation_chain_results_detail.csv'
ROUND1I_RESULTS = PROJECT_ROOT / 'results' / '12_round1i_run' / 'round1i_operator_results_detail.csv'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)


In [ ]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

round1c_rows = read_csv(ROUND1C_TABLE)
round1g_rows = read_csv(ROUND1G_RESULTS)
round1i_rows = read_csv(ROUND1I_RESULTS)

base_2639 = [r for r in round1c_rows if r['target_task_id'] == 'wiki_dev_2639']
print('round1c wiki_dev_2639 rows =', len(base_2639))
for r in base_2639:
    print(r['condition'], r['split'], r['em'], r['source_set_id'], r['role_reason'])


In [ ]:
patch_rows = []

# Round 1g repaired episodic rows
for r in round1g_rows:
    if r['target_task_id'] == 'wiki_dev_2639' and r['condition'] == 'episodic_trace':
        patch_rows.append({
            'source_round': 'round1g',
            'condition': r['condition'],
            'split': r['split'],
            'source_set_id': r['source_set_id'],
            'em': r['em'],
            'f1': r['f1'],
            'pred_answer': r['pred_answer'],
            'failure_status': r['failure_status'],
            'memory_reference_type': r['memory_reference_type'],
            'reasoning_present': r['reasoning_present'],
            'final_answer_present': r['final_answer_present'],
            'note': 'subtype-aware rerun result'
        })

# Round 1i repaired relevant consolidation and updated irrelevant consolidation
for r in round1i_rows:
    cond = r['condition']
    if cond == 'operator_repaired_relevant_consolidation':
        patch_rows.append({
            'source_round': 'round1i',
            'condition': 'cross_episode_consolidation',
            'split': 'relevant',
            'source_set_id': r['source_set_id'],
            'em': r['em'],
            'f1': r['f1'],
            'pred_answer': r['pred_answer'],
            'failure_status': r['failure_status'],
            'memory_reference_type': r['memory_reference_type'],
            'reasoning_present': r['reasoning_present'],
            'final_answer_present': r['final_answer_present'],
            'note': 'operator-repaired relevant consolidation'
        })
    elif cond == 'irrelevant_consolidation':
        patch_rows.append({
            'source_round': 'round1i',
            'condition': 'cross_episode_consolidation',
            'split': 'irrelevant',
            'source_set_id': r['source_set_id'],
            'em': r['em'],
            'f1': r['f1'],
            'pred_answer': r['pred_answer'],
            'failure_status': r['failure_status'],
            'memory_reference_type': r['memory_reference_type'],
            'reasoning_present': r['reasoning_present'],
            'final_answer_present': r['final_answer_present'],
            'note': 'subtype-aware irrelevant consolidation after operator repair round'
        })

patch_rows = sorted(patch_rows, key=lambda x: (x['condition'], x['split']))
for r in patch_rows:
    print(r)


In [ ]:
patch_csv = RESULTS_DIR / 'round1j_wiki_dev_2639_patch_rows.csv'
fieldnames = ['source_round', 'condition', 'split', 'source_set_id', 'em', 'f1', 'pred_answer', 'failure_status', 'memory_reference_type', 'reasoning_present', 'final_answer_present', 'note']
with patch_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(patch_rows)
print('wrote', patch_csv)


In [ ]:
before = {(r['condition'], r['split']): r for r in base_2639 if r['condition'] != 'no_memory'}
after = {(r['condition'], r['split']): r for r in patch_rows}

summary_lines = []
summary_lines.append('# Round 1j Patchback Summary')
summary_lines.append('')
summary_lines.append('Date: 2026-04-11')
summary_lines.append('')
summary_lines.append('## Objective')
summary_lines.append('')
summary_lines.append('Patch the repaired relation-chain evidence for `wiki_dev_2639` back into the role-aware interpretation layer.')
summary_lines.append('')
summary_lines.append('## Why This Patchback Is Needed')
summary_lines.append('')
summary_lines.append('- The original Round 1c table still reflects the coarse-bridge / pre-repair behavior for `wiki_dev_2639`.')
summary_lines.append('- After Round 1g and Round 1i, that case no longer supports the same diagnosis.')
summary_lines.append('')
summary_lines.append('## Before vs After on `wiki_dev_2639`')
summary_lines.append('')
summary_lines.append('| condition | split | Round 1c status | patched status | interpretation |')
summary_lines.append('|---|---|---|---|---|')
for key in [('episodic_trace', 'relevant'), ('episodic_trace', 'irrelevant'), ('cross_episode_consolidation', 'relevant'), ('cross_episode_consolidation', 'irrelevant')]:
    b = before.get(key)
    a = after.get(key)
    if not b or not a:
        continue
    b_status = f"EM={b['em']} / {b['pred_answer']}"
    a_status = f"EM={a['em']} / {a['pred_answer']}"
    if key == ('episodic_trace', 'relevant'):
        interp = 'relevant episodic is now recovered by subtype-aware routing'
    elif key == ('cross_episode_consolidation', 'relevant'):
        interp = 'relevant consolidation is now recovered after operator repair'
    elif key[1] == 'irrelevant':
        interp = 'irrelevant memory remains harmful / refusal-prone'
    else:
        interp = 'updated'
    summary_lines.append(f"| {key[0]} | {key[1]} | {b_status} | {a_status} | {interp} |")
summary_lines.append('')
summary_lines.append('## Updated Interpretation')
summary_lines.append('')
summary_lines.append('- `wiki_dev_2639` should no longer be described as a case where relevant bridge memory harms an originally correct baseline.')
summary_lines.append('- After subtype-aware rerouting and operator repair, the relevant memory path becomes recoverable in both `episodic_trace` and `cross_episode_consolidation`.')
summary_lines.append('- The more accurate interpretation is now: this case exposed a false negative caused by coarse pairing granularity plus incomplete operator guidance inside consolidation.')
summary_lines.append('')
summary_lines.append('## Implication for Round 1 Synthesis')
summary_lines.append('')
summary_lines.append('- The strongest remaining diagnostic message is no longer “relevant memory can hurt on this bridge case.”')
summary_lines.append('- The stronger message is “selective transfer claims are highly sensitive to pairing granularity and to how abstract memory operationalizes relation operators.”')
summary_lines.append('')
summary_lines.append('## Next Step')
summary_lines.append('')
summary_lines.append('- Use this patched interpretation in the final Round 1 synthesis instead of the original Round 1c wording for `wiki_dev_2639`.')

summary_md = RESULTS_DIR / 'round1j_patchback_summary.md'
summary_md.write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
print('wrote', summary_md)
